# 05. W7 External Features, Optimization, and SHAP Analysis

**Objective**: Integrate weather and holiday features, quantify auxiliary impact (VG Node 1), perform Bayesian hyperparameter optimization, and apply SHAP explainability.

**Week 7 Tasks**:
- [x] External Feature Integration (weather + holidays)
- [x] Ablation Study and VG Node 1 Impact Quantification
- [x] Optuna Hyperparameter Tuning
- [x] SHAP Explainability Analysis
- [x] Residual Diagnostics
- [x] Model Persistence


## 1. Setup and Data Loading


In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathimport sysimport warningswarnings.filterwarnings('ignore')# Add project root to pathPROJECT_ROOT = Path.cwd().parentif str(PROJECT_ROOT) not in sys.path:    sys.path.append(str(PROJECT_ROOT))from src.data_processor import DataProcessorfrom src.utils import logger, DATA_PATHS, convert_to_swedish_time, load_data, save_datafrom src.models import (    RandomForestForecaster,    XGBoostForecaster,    LightGBMForecaster,    compare_models,    create_train_test_split)sns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load W7 pre-computed features (with weather and holidays)try:    df = load_data('processed', 'se3_features_w7.parquet')    print(f"Loaded existing W7 features: {df.shape}")except FileNotFoundError:    print("Generating W7 features from scratch...")    processor = DataProcessor(target_area='SE3')    df = processor.prepare_features()    # Weather integration fallback    import requests    lat, lon = 59.3293, 18.0686    url = 'https://archive-api.open-meteo.com/v1/archive'    params = {        'latitude': lat, 'longitude': lon,        'start_date': '2024-01-01', 'end_date': '2025-12-31',        'hourly': ['temperature_2m', 'windspeed_10m'],        'timezone': 'Europe/Stockholm'    }    r = requests.get(url, params=params, timeout=120)    r.raise_for_status()    data = r.json()    weather_df = pd.DataFrame({        'timestamp': pd.to_datetime(data['hourly']['time']),        'temperature': data['hourly']['temperature_2m'],        'wind_speed': data['hourly']['windspeed_10m']    })    weather_df = convert_to_swedish_time(weather_df, 'timestamp')    df = processor.add_weather_features(df, smhi_df=weather_df)    processor.save_processed_data(df, 'se3_features_w7.parquet')    print(f"Generated and saved W7 features: {df.shape}")df.head()

## 2. Feature Inspection

Inspect the expanded feature set to confirm weather and holiday features were correctly integrated.


In [ ]:
print(f"Data range: {df['timestamp'].min()} to {df['timestamp'].max()}")print(f"Features: {len(df.columns)} columns")print("\nColumns:")for i, c in enumerate(df.columns, 1):    print(f"  {i:2d}. {c}")# Categorize featuresall_features = [c for c in df.columns if c not in ['timestamp', 'value']]weather_features = [f for f in all_features if any(x in f.lower() for x in ['temp', 'wind'])]holiday_features = [f for f in all_features if 'holiday' in f.lower()]cyclical_features = [f for f in all_features if any(x in f for x in ['_sin', '_cos'])]core_features = [f for f in all_features if f not in weather_features + holiday_features + cyclical_features]print("\nFeature breakdown:")print(f"  Core:      {len(core_features)} features")print(f"  Cyclical:  {len(cyclical_features)} features")print(f"  Holiday:   {len(holiday_features)} features")print(f"  Weather:   {len(weather_features)} features")print(f"  Total:     {len(all_features)} features")

## 3. Train/Test Split


In [ ]:
# 80/20 chronological splitX_train, y_train, X_test, y_test = create_train_test_split(    df,    target_col='value',    train_ratio=0.8,    drop_cols=[])print("\n" + "="*50)print("DATASET SPLIT SUMMARY")print("="*50)print(f"Training set:   {len(X_train):,} samples ({len(X_train)/len(df)*100:.1f}%)")print(f"Test set:       {len(X_test):,} samples ({len(X_test)/len(df)*100:.1f}%)")print(f"\nFeatures:       {X_train.shape[1]}")print(f"\nTarget stats:")print(f"  Train mean: {y_train.mean():.2f} SEK/MWh")print(f"  Test mean:  {y_test.mean():.2f} SEK/MWh")

## 4. Baseline Model Training (Full Features)

Train all three models on the complete feature dataset.


In [ ]:
# Initialize all three modelsmodels = {    'Random Forest': RandomForestForecaster(        n_estimators=100,        max_depth=10,        random_state=42    ),    'XGBoost': XGBoostForecaster(        n_estimators=1000,        learning_rate=0.01,        max_depth=6,        subsample=0.8,        colsample_bytree=0.8,        random_state=42    ),    'LightGBM': LightGBMForecaster(        n_estimators=1000,        learning_rate=0.01,        max_depth=6,        num_leaves=31,        subsample=0.8,        colsample_bytree=0.8,        random_state=42    )}# Trainfor name, model in models.items():    print(f"\nTraining {name}...")    model.fit(X_train, y_train)    print(f"  Training complete.")

## 5. Model Evaluation and Comparison


In [ ]:
# Generate predictions for each modelpredictions = {}results = []for name, model in models.items():    preds = model.predict(X_test)    predictions[name] = preds    metrics = model.evaluate(X_test, y_test, verbose=False)    results.append({        'Model': name,        'MAE': metrics['mae'],        'RMSE': metrics['rmse'],        'MAPE (%)': metrics['mape']    })# Display comparison tableresults_df = pd.DataFrame(results)print("\n" + "="*60)print("MODEL PERFORMANCE COMPARISON (Full Features)")print("="*60)print(results_df.to_string(index=False))print("="*60)

In [ ]:
# Visualization: Actual vs Predictedfig, axes = plt.subplots(1, 3, figsize=(18, 5))for idx, (name, preds) in enumerate(predictions.items()):    ax = axes[idx]    ax.scatter(y_test.iloc[::10], preds[::10], alpha=0.3, s=1)    min_val = min(y_test.min(), preds.min())    max_val = max(y_test.max(), preds.max())    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect')    ax.set_xlabel('Actual Price (SEK/MWh)')    ax.set_ylabel('Predicted Price (SEK/MWh)')    ax.set_title(f'{name}')    ax.legend()plt.tight_layout()plt.show()

In [ ]:
# Time series prediction visualization (sample of test set)sample_size = min(168, len(y_test))sample_idx = slice(0, sample_size)fig, ax = plt.subplots(figsize=(16, 6))ax.plot(y_test.iloc[sample_idx].values, label='Actual', color='black', linewidth=2)colors = ['#2ecc71', '#3498db', '#e74c3c']for (name, preds), color in zip(predictions.items(), colors):    ax.plot(preds[sample_idx], label=name, alpha=0.7, linewidth=1.5, color=color)ax.set_xlabel('Hours')ax.set_ylabel('Price (SEK/MWh)')ax.set_title('Test Set Predictions - First Week')ax.legend()plt.show()

## 6. Feature Importance Analysis

Analyze feature importance for all three models on the expanded feature set.


In [ ]:
# Get feature importance from all modelsfig, axes = plt.subplots(1, 3, figsize=(20, 8))for idx, (name, model) in enumerate(models.items()):    importance = model.feature_importance()    if importance is not None:        top_features = importance.head(15)        ax = axes[idx]        top_features.plot(kind='barh', ax=ax, color='steelblue')        ax.set_title(f'{name}\nTop 15 Feature Importance')        ax.set_xlabel('Importance')        ax.invert_yaxis()plt.tight_layout()plt.show()

In [ ]:
# Detailed XGBoost feature importance (for VG report)xgb_model = models['XGBoost']xgb_importance = xgb_model.feature_importance()print("\n" + "="*60)print("XGBOOST FEATURE IMPORTANCE RANKING (Top 20)")print("="*60)for rank, (feature, importance) in enumerate(xgb_importance.head(20).items(), 1):    marker = ""    if 'wind' in feature.lower():        marker = " [WEATHER]"    elif 'temp' in feature.lower():        marker = " [WEATHER]"    elif 'holiday' in feature.lower():        marker = " [HOLIDAY]"    elif '_sin' in feature or '_cos' in feature:        marker = " [CYCLICAL]"    print(f"{rank:2d}. {feature:<30} {importance:.4f}{marker}")print("="*60)

## 7. VG Node 1: Auxiliary Feature Impact Analysis

Quantify the impact of weather, holiday, and cyclical auxiliary features on model performance using a controlled ablation study.


In [ ]:
# Define feature sets for ablation studyall_features = list(X_train.columns)weather_features = [f for f in all_features if any(x in f.lower() for x in ['temp', 'wind'])]holiday_features = [f for f in all_features if 'holiday' in f.lower()]cyclical_features = [f for f in all_features if any(x in f for x in ['_sin', '_cos'])]print("Feature Categories:")print(f"  Weather features:   {len(weather_features)} ({', '.join(weather_features)})")print(f"  Holiday features:   {len(holiday_features)} ({', '.join(holiday_features)})")print(f"  Cyclical features:  {len(cyclical_features)} ({', '.join(cyclical_features)})")print(f"  Core features:      {len([f for f in all_features if f not in weather_features + holiday_features + cyclical_features])}")

In [ ]:
# Ablation study: Train XGBoost with different feature setsfeature_sets = {    'All Features': all_features,    'No Weather': [f for f in all_features if f not in weather_features],    'No Holidays': [f for f in all_features if f not in holiday_features],    'No Weather + No Holidays': [f for f in all_features if f not in weather_features + holiday_features],    'Core Only (No Aux)': [f for f in all_features if f not in weather_features + holiday_features + cyclical_features]}ablation_results = []for set_name, cols in feature_sets.items():    model = XGBoostForecaster(        n_estimators=1000, learning_rate=0.01, max_depth=6,        subsample=0.8, colsample_bytree=0.8, random_state=42    )    model.fit(X_train[cols], y_train)    metrics = model.evaluate(X_test[cols], y_test, verbose=False)    ablation_results.append({        'Feature Set': set_name,        'N_Features': len(cols),        'MAE': metrics['mae'],        'RMSE': metrics['rmse']    })    print(f"{set_name}: MAE={metrics['mae']:.4f}, RMSE={metrics['rmse']:.4f}")ablation_df = pd.DataFrame(ablation_results)ablation_df

In [ ]:
# Calculate impact of auxiliary featuresbaseline_mae = ablation_df[ablation_df['Feature Set'] == 'All Features']['MAE'].values[0]no_weather_mae = ablation_df[ablation_df['Feature Set'] == 'No Weather']['MAE'].values[0]no_holidays_mae = ablation_df[ablation_df['Feature Set'] == 'No Holidays']['MAE'].values[0]no_aux_mae = ablation_df[ablation_df['Feature Set'] == 'No Weather + No Holidays']['MAE'].values[0]weather_impact = no_weather_mae - baseline_maeholiday_impact = no_holidays_mae - baseline_maecombined_aux_impact = no_aux_mae - baseline_maeprint("\n" + "="*60)print("ABLATION IMPACT SUMMARY")print("="*60)print(f"Baseline (All Features) MAE:        {baseline_mae:.4f}")print(f"Weather Impact:                     {weather_impact:+.4f} ({weather_impact/baseline_mae*100:+.2f}%)")print(f"Holiday Impact:                     {holiday_impact:+.4f} ({holiday_impact/baseline_mae*100:+.2f}%)")print(f"Combined Aux Impact:                {combined_aux_impact:+.4f} ({combined_aux_impact/baseline_mae*100:+.2f}%)")print("="*60)

In [ ]:
# Visualization: Ablation study resultsfig, axes = plt.subplots(1, 2, figsize=(16, 6))# MAE comparisonax1 = axes[0]colors = ['#2ecc71' if 'All' in x else '#3498db' if 'No' in x else '#95a5a6' for x in ablation_df['Feature Set']]bars = ax1.barh(ablation_df['Feature Set'], ablation_df['MAE'], color=colors)ax1.set_xlabel('MAE (SEK/MWh)')ax1.set_title('Model Performance by Feature Set\n(Lower is Better)')ax1.invert_yaxis()for bar, val in zip(bars, ablation_df['MAE']):    ax1.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')# Feature count vs MAEax2 = axes[1]ax2.scatter(ablation_df['N_Features'], ablation_df['MAE'], s=200, c=colors, edgecolors='black')for _, row in ablation_df.iterrows():    ax2.annotate(row['Feature Set'], (row['N_Features'], row['MAE']),                 textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)ax2.set_xlabel('Number of Features')ax2.set_ylabel('MAE (SEK/MWh)')ax2.set_title('Feature Count vs Performance')plt.tight_layout()plt.show()

## 8. Cross-Validation Robustness Test

Re-run 5-fold time-series CV on the full W7 feature set.


In [ ]:
# 5-fold time-series cross-validationprint("\n" + "="*60)print("5-FOLD TIME-SERIES CROSS-VALIDATION")print("="*60)X_full = pd.concat([X_train, X_test], ignore_index=True)y_full = pd.concat([y_train, y_test], ignore_index=True)cv_results = {}for name, model in models.items():    print(f"\n{'-'*60}")    cv_summary = model.cross_validate(X_full, y_full, n_splits=5)    cv_results[name] = cv_summary# Summary tableprint("\n" + "="*60)print("CROSS-VALIDATION SUMMARY")print("="*60)cv_df = pd.DataFrame([    {        'Model': name,        'CV MAE Mean': r['cv_mae_mean'],        'CV MAE Std': r['cv_mae_std'],        'CV RMSE Mean': r['cv_rmse_mean']    } for name, r in cv_results.items()])print(cv_df.to_string(index=False))print("="*60)

## 9. Optuna Hyperparameter Tuning

Use Bayesian optimization to find the best XGBoost regularization parameters.


In [ ]:
import optuna# Use validation set from end of training dataval_size = int(len(X_train) * 0.1)X_tr = X_train.iloc[:-val_size]y_tr = y_train.iloc[:-val_size]X_val = X_train.iloc[-val_size:]y_val = y_train.iloc[-val_size:]def objective(trial):    params = {        'n_estimators': 1000,        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),        'max_depth': trial.suggest_int('max_depth', 3, 10),        'subsample': trial.suggest_float('subsample', 0.6, 1.0),        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),    }    model = XGBoostForecaster(**params)    model.fit(X_tr, y_tr, X_val=X_val, y_val=y_val)    preds = model.predict(X_val)    mae = np.mean(np.abs(y_val.values - preds))    return maestudy = optuna.create_study(direction='minimize')study.optimize(objective, n_trials=50, show_progress_bar=True)print(f"\nBest trial MAE: {study.best_value:.4f}")print("Best parameters:")for k, v in study.best_params.items():    print(f"  {k}: {v}")

In [ ]:
# Train final optimized model with more treesbest_params = study.best_params.copy()best_params['n_estimators'] = 2000xgb_optimized = XGBoostForecaster(name='XGBoost_Optimized', **best_params)xgb_optimized.fit(X_train, y_train)opt_metrics = xgb_optimized.evaluate(X_test, y_test)print("\n" + "="*60)print("OPTIMIZED MODEL COMPARISON")print("="*60)base_mae = results_df[results_df['Model']=='XGBoost']['MAE'].values[0]print(f"Baseline XGBoost MAE:  {base_mae:.4f}")print(f"Optimized XGBoost MAE: {opt_metrics['mae']:.4f}")print(f"Improvement:           {base_mae - opt_metrics['mae']:.4f}")print("="*60)

## 10. SHAP Explainability Analysis

Understand how each feature drives predictions for the optimized model.


In [ ]:
import shapexplainer = shap.TreeExplainer(xgb_optimized.model)X_sample = X_test.sample(n=min(1000, len(X_test)), random_state=42)shap_values = explainer.shap_values(X_sample)# Mean absolute SHAP valuesmean_shap = pd.DataFrame({    'feature': X_sample.columns,    'mean_abs_shap': np.abs(shap_values).mean(axis=0)}).sort_values('mean_abs_shap', ascending=False)print("Top 10 SHAP features:")for _, row in mean_shap.head(10).iterrows():    print(f"  {row['feature']}: {row['mean_abs_shap']:.4f}")

In [ ]:
# SHAP summary plot (beeswarm)shap.summary_plot(shap_values, X_sample, max_display=15, show=False)plt.title('SHAP Feature Importance (Optimized XGBoost)')plt.tight_layout()plt.show()

In [ ]:
# SHAP bar plotshap.summary_plot(shap_values, X_sample, plot_type='bar', max_display=15, show=False)plt.title('Mean |SHAP| Value (Optimized XGBoost)')plt.tight_layout()plt.show()

## 11. Residual and Diagnostic Analysis

Identify systematic prediction errors by time of day and month.


In [ ]:
# Build residual dataframetest_df = df.iloc[len(X_train):].copy()test_df['pred'] = xgb_optimized.predict(X_test)test_df['residual'] = test_df['value'] - test_df['pred']test_df['abs_err'] = np.abs(test_df['residual'])print("Residual Summary:")print(f"  Mean residual:       {test_df['residual'].mean():.4f}")print(f"  Median residual:     {test_df['residual'].median():.4f}")print(f"  Std residual:        {test_df['residual'].std():.4f}")print(f"  Max absolute error:  {test_df['abs_err'].max():.4f}")print(f"  95th pct abs error:  {np.percentile(test_df['abs_err'], 95):.4f}")

In [ ]:
# Hourly and monthly error patternsfig, axes = plt.subplots(1, 2, figsize=(16, 5))hourly_err = test_df.groupby('hour')['abs_err'].mean().sort_index()axes[0].bar(hourly_err.index, hourly_err.values, color='steelblue')axes[0].set_xlabel('Hour of Day')axes[0].set_ylabel('Mean Absolute Error')axes[0].set_title('MAE by Hour of Day')monthly_err = test_df.groupby('month')['abs_err'].mean().sort_index()axes[1].bar(monthly_err.index, monthly_err.values, color='coral')axes[1].set_xlabel('Month')axes[1].set_ylabel('Mean Absolute Error')axes[1].set_title('MAE by Month')plt.tight_layout()plt.show()

## 12. Model Persistence

Save the optimized models and artifacts for Week 8 deployment.


In [ ]:
print("\n" + "="*60)print("SAVING MODELS")print("="*60)saved = {}saved['XGBoost_Optimized'] = xgb_optimized.save('w7_xgboost_optimized.pkl')saved['XGBoost_Baseline'] = models['XGBoost'].save('w7_xgboost_baseline.pkl')saved['LightGBM_Baseline'] = models['LightGBM'].save('w7_lightgbm_baseline.pkl')for name, path in saved.items():    print(f"{name}: {path.name}")# Save artifactsresults_dir = DATA_PATHS['reports'] / 'w7_results'results_dir.mkdir(parents=True, exist_ok=True)results_df.to_csv(results_dir / 'model_comparison.csv', index=False)ablation_df.to_csv(results_dir / 'ablation_study.csv', index=False)mean_shap.to_csv(results_dir / 'shap_top_features.csv', index=False)print(f"\nArtifacts saved to {results_dir}")print("="*60)

## 13. Summary and Next Steps


In [ ]:
print("\n" + "="*70)print("W7 DEVELOPMENT SUMMARY")print("="*70)print("\nCompleted Tasks:")print("  1. External Feature Integration")print("     - Weather: temperature, wind_speed, lags, squared wind")print("     - Holidays: is_holiday, is_pre_holiday")print("     - Cyclical: hour_sin/cos, month_sin/cos, dow_sin/cos")print("\n  2. Baseline Comparison (Full Features)")for _, row in results_df.iterrows():    print(f"     {row['Model']}: MAE={row['MAE']:.4f}")print(f"\n  3. Optuna Tuning Results")print(f"     Optimized XGBoost MAE: {opt_metrics['mae']:.4f}")base_mae = results_df[results_df['Model']=='XGBoost']['MAE'].values[0]print(f"     Improvement over baseline: {base_mae - opt_metrics['mae']:.4f}")print(f"\n  4. VG Node 1: Auxiliary Feature Impact")print(f"     Weather impact:  {weather_impact:+.4f} ({weather_impact/baseline_mae*100:+.2f}%)")print(f"     Holiday impact:  {holiday_impact:+.4f} ({holiday_impact/baseline_mae*100:+.2f}%)")print("\n  5. SHAP Top 3 Features")for _, row in mean_shap.head(3).iterrows():    print(f"     {row['feature']}: {row['mean_abs_shap']:.4f}")print("\n Next Week (W8) Targets:")print("  - Streamlit app with interactive forecasting")print("  - Model deployment artifacts")print("  - Final project report and VG documentation")print("="*70)